# M6.0a Primer: Emf, Motional Emf and Faraday's Law

Book pointer: chapter 14, sections 14.1 (emf), 14.2 (motional emf), 14.3 (Faraday's law), 14.4 (Faraday's law on differential form).

First of four primers for module 6. Modules 2 to 5 treated fields that do not change in time: charges at rest made $\mathbf{E}$, steady currents made $\mathbf{B}$, and the two never met. This primer is where they meet. A magnet pushed through a coil makes a current flow, although there is no battery. A coil whose current changes makes a current flow in a second coil that it does not touch. Both are the same law, Faraday's law, and it is the first of the two couplings between $\mathbf{E}$ and $\mathbf{B}$ that turn them into a single electromagnetic field. The second coupling, the displacement current, comes in primer 00c. Every result here is derived by hand and then reproduced numerically with `emlib`, so that you can follow notebook M6.1 without the book.

## 1. Warm-up

Answer on paper before reading on. Short answers are in section 7.

1. What is an emf, and how does it differ from a potential difference?
2. A metal rod of length $L$ moves at speed $v$ perpendicular to a uniform field $\mathbf{B}$, sliding on two rails joined by a resistor. What emf drives the current, and where does the energy dissipated in the resistor come from?
3. State Faraday's law. The flux of which field, through which surface?
4. The north pole of a bar magnet approaches a wire loop. Seen from the magnet, which way does the induced current circulate, and why?
5. Why can the induced electric field not be written as minus the gradient of a potential?
6. Write Faraday's law in differential form and show how it reduces to the electrostatic statement $\nabla\times\mathbf{E} = 0$.
7. Why are transformer cores built from thin insulated sheets rather than solid iron?

## 2. The ideas

**Emf.** A steady current around a closed circuit needs something that pushes charge all the way round. The electrostatic field cannot do it: its line integral around any closed loop is zero (module 1), so whatever work it does on a charge going down one side it takes back on the way up the other. The *electromotive force*, or emf, $\mathcal{E}$ of a circuit is the work done per unit charge by all the forces acting on the carriers, taken once around the circuit. Only non-electrostatic forces contribute: the chemistry of a battery, the magnetic force on carriers in a moving wire, or an electric field that is not conservative. The name is old and misleading. An emf is not a force; it is a work per charge and is measured in volts.

**Two ways to make an emf with magnetism.** Move a wire through a magnetic field and the carriers in the wire move with it, so they feel the magnetic force $q\mathbf{v}\times\mathbf{B}$ along the wire. That is *motional emf*, and it needs nothing beyond the Lorentz force of module 4. The second way is stranger. Keep the wire still and change the field instead, by moving a magnet or changing the current in a nearby coil. The carriers are at rest, so the magnetic force on them is zero, yet a current still flows. Something must push them, and the only candidate is an electric field. A changing magnetic field creates an electric field. That is new physics, and it is what Faraday discovered.

**The flux rule.** Both cases obey one rule. Let $\Phi_B$ be the magnetic flux through the circuit, the integral of $\mathbf{B}\cdot d\mathbf{S}$ over a surface bounded by the wire. Then
$$\mathcal{E} = -\frac{d\Phi_B}{dt}.$$
It does not matter whether the flux changes because the circuit moves, because it changes shape, or because the field changes. For motional emf the rule is a theorem that follows from the Lorentz force. For a changing field it is a law of nature. That one formula covers both is a coincidence in Newtonian physics, and it is one of the observations that led Einstein to special relativity: whether a magnet moves past a wire or the wire moves past the magnet, the current is the same.

**Lenz's law.** The minus sign says that the induced emf drives a current whose own magnetic field opposes the change in flux. Push a magnet towards a loop and the loop's current makes a field that pushes back on the magnet. Pull it away and the current reverses and holds the magnet back. If the sign were the other way, the induced current would increase the flux, which would increase the current, and a nudge would run away into unlimited energy. Lenz's law is energy conservation applied to induction.

**The induced electric field is not conservative.** Around a loop in a changing magnetic field, $\oint\mathbf{E}\cdot d\mathbf{l} = -d\Phi_B/dt \neq 0$. A field with a non-zero circulation cannot be the gradient of a potential, because the line integral of a gradient around a closed loop is always zero. So "voltage between two points" loses its unique meaning in a changing magnetic field: it depends on the path between the points. A voltmeter's leads form part of the path, and moving them around changes the reading. Circuit theory in module 5 works because the changing flux is confined inside components such as inductors, which are then described by the voltage $L\,dI/dt$ across their terminals.

**The differential form.** Stokes' theorem (primer M1.0d) turns the loop statement into a statement at a point:
$$\nabla\times\mathbf{E} = -\frac{\partial\mathbf{B}}{\partial t}.$$
In electrostatics $\partial\mathbf{B}/\partial t = 0$ and this becomes $\nabla\times\mathbf{E} = 0$, the property that made the potential $V$ possible. Faraday's law is therefore not an add-on. It replaces one of the two laws of electrostatics. Together with $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$, $\nabla\cdot\mathbf{B} = 0$ and Ampere's law with the displacement current (primer 00c), it completes Maxwell's equations.

**Eddy currents and machines.** The induced field does not need a wire. In any conductor it drives currents that circulate in closed paths, called eddy currents. They heat the conductor and, by Lenz's law, brake any motion that causes them. Generators turn mechanical rotation into emf by turning a coil in a field. Transformers couple two coils through a shared changing flux. Induction cooktops, metal detectors, magnetic brakes and the laminated cores of every transformer are all applications of the same law.

**What notebook M6.1 does.** It computes the flux through a loop that moves through a non-uniform field, differentiates the flux numerically to get the emf, and checks the answer against the line integral of $\mathbf{v}\times\mathbf{B}$ around the moving loop. It then turns a coil in a field to make a generator, drops a magnet through a loop, and checks the sign of every result against Lenz's law. Everything those calculations need is set up below.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import biot_savart as bs
from emlib import fields
from emlib import induction as ind
from emlib.biot_savart import MU0

plt.rcParams["figure.figsize"] = (6, 4)
np.set_printoptions(precision=4, suppress=True)

### 3.1 Emf (section 14.1)

Take a charge $q$ once around a closed circuit and add up the work done on it by every force per unit charge $\mathbf{f}$ that acts along the way. That sum is the emf,
$$\mathcal{E} = \oint \mathbf{f}\cdot d\mathbf{l}.$$
The electrostatic part of $\mathbf{f}$, the field $\mathbf{E}_{\text{es}}$ made by charges at rest, contributes nothing, because $\oint\mathbf{E}_{\text{es}}\cdot d\mathbf{l} = 0$ for every loop. What survives is the work of the non-electrostatic forces. Inside a battery that is a chemical force that carries positive charge from the negative to the positive terminal against the electrostatic field. Inside a moving wire it is the magnetic force on the carriers. In a changing magnetic field it is an induced electric field that is not conservative.

**Emf versus potential difference.** A potential difference $V_b - V_a = -\int_a^b\mathbf{E}_{\text{es}}\cdot d\mathbf{l}$ is defined between two points and does not depend on the path. An emf is defined for a closed path and is a property of the whole loop. In an ideal battery with no current flowing, the chemical force exactly balances the electrostatic force inside, so the terminal voltage equals the emf. When current flows, part of the emf is used up driving current through the battery's internal resistance (primer M5.0b) and the terminal voltage drops below the emf.

A common confusion is to call the emf "the voltage of the battery" and then try to find a voltage for an induced emf as well. For a battery that is harmless. For induction it fails: section 3.3 shows a loop with a non-zero emf around it but no point at which a potential could be defined consistently.

**Worked example 1 (what an emf does).** A 9 V battery drives 0.1 A through a resistor for one hour. The charge passed is $Q = It = 0.1\times3600 = 360\,$C. The work done by the battery's emf is $\mathcal{E}Q = 9\times360 = 3240\,$J, all of which ends up as heat in the resistor and the battery. The power is $\mathcal{E}I = 0.9\,$W. Now suppose the battery is removed and the wire is closed into a loop in a static electric field made by fixed charges. The emf of the loop is $\oint\mathbf{E}_{\text{es}}\cdot d\mathbf{l} = 0$ whatever the charges are, so no steady current can flow. The code checks this with the superposition field of three charges and a wobbly closed loop.

In [ ]:
Q_passed = 0.1 * 3600
print(f"charge passed {Q_passed:.0f} C, work by the emf {9 * Q_passed:.0f} J, power {9 * 0.1:.1f} W")

# The electrostatic field of three charges has zero circulation around any closed loop.
src = np.array([[0.3, 0.1, 0.0], [-0.2, 0.4, 0.1], [0.1, -0.3, -0.2]])
q = np.array([2e-9, -3e-9, 1.5e-9])
t = np.linspace(0, 2 * np.pi, 4001)
loop = np.stack([0.5 * np.cos(t) * (1 + 0.3 * np.sin(3 * t)), 0.5 * np.sin(t), 0.1 * np.cos(2 * t)], axis=-1)
circ = fields.line_integral(lambda p: fields.point_charges_E(src, q, p), loop)
half = fields.line_integral(lambda p: fields.point_charges_E(src, q, p), loop[:2001])
print(f"circulation of the electrostatic field around the loop: {circ:.2e} V")
print(f"line integral along half the loop, for scale: {half:.2f} V")

### 3.2 Motional emf (section 14.2)

**The rod on rails.** Two parallel rails a distance $L$ apart lie in a uniform magnetic field $\mathbf{B}$ pointing out of the page. A resistor $R$ joins the rails at one end, and a conducting rod lies across them and slides to the right at speed $v$. The carriers in the rod move with it, so each feels the magnetic force $q\mathbf{v}\times\mathbf{B}$. With $\mathbf{v}$ to the right and $\mathbf{B}$ out of the page, $\mathbf{v}\times\mathbf{B}$ points down along the rod for a positive carrier. The force per unit charge has magnitude $vB$ and acts along the full length $L$ of the rod, so the work per charge around the circuit is
$$\mathcal{E} = BLv.$$
The rest of the circuit is at rest and contributes nothing. A current $I = BLv/R$ flows, down the rod and back up through the resistor.

**The flux rule gives the same answer.** The circuit encloses an area $Lx$, where $x$ is the distance from the resistor to the rod, so $\Phi_B = BLx$ and $d\Phi_B/dt = BLv$. The emf has the same size, and the sign of the flux rule gives the same direction, as the next paragraph shows.

**The retarding force and the power balance.** The rod now carries a current $I$ in the field, so it feels the force $I\mathbf{L}\times\mathbf{B}$ (primer M4.0a). With the current flowing down the rod, the force points to the left, against the motion. Its size is $F = BIL = B^2L^2v/R$. To keep the rod moving at constant speed, an outside agent must push with that force and does work at the rate
$$Fv = \frac{B^2L^2v^2}{R} = \frac{(BLv)^2}{R} = I^2R.$$
Every joule of heat in the resistor is a joule of mechanical work done on the rod. The magnetic field does no work, as it never does (primer M4.0a); it only redirects the push of the outside agent into the current. If the sign of the induced current were reversed, the force would push the rod forward and the rod would accelerate on its own, which is the runaway that Lenz's law forbids.

**Worked example 2 (numbers).** $B = 0.5\,$T, $L = 0.2\,$m, $v = 2\,$m/s, $R = 0.1\,\Omega$. Emf $= 0.5\times0.2\times2 = 0.2\,$V. Current $= 0.2/0.1 = 2\,$A. Force $= BIL = 0.5\times2\times0.2 = 0.2\,$N against the motion. Mechanical power $= 0.2\times2 = 0.4\,$W, and $I^2R = 4\times0.1 = 0.4\,$W.

**The general form.** For any circuit whose pieces move with velocity $\mathbf{v}(\mathbf{r})$ through a static field,
$$\mathcal{E} = \oint(\mathbf{v}\times\mathbf{B})\cdot d\mathbf{l}.$$
Only the parts of the circuit that move across field lines contribute. A segment moving along its own length, or parallel to $\mathbf{B}$, gives nothing.

**The generator.** A flat coil of $N$ turns and area $A$ rotates at angular speed $\omega$ about an axis perpendicular to a uniform field $B$. The angle between the coil's normal and $\mathbf{B}$ is $\omega t$, so the flux through each turn is $BA\cos\omega t$ and
$$\mathcal{E} = -N\frac{d}{dt}(BA\cos\omega t) = NBA\omega\sin\omega t.$$
The emf alternates, with peak value $NBA\omega$. Every power station generator works this way, with the field usually made by an electromagnet on the rotating part and the coils fixed around it.

**Worked example 3 (a 50 Hz generator coil).** $N = 100$ turns, $A = 0.01\,\mathrm{m^2}$, $B = 0.5\,$T, $\omega = 2\pi\times50 = 314.2\,\mathrm{s^{-1}}$. Peak emf $= 100\times0.5\times0.01\times314.2 = 157.1\,$V, root-mean-square $157.1/\sqrt2 = 111.1\,$V.

In [ ]:
B, L, v, R = 0.5, 0.2, 2.0, 0.1
emf = B * L * v; I = emf / R; F = B * I * L
print(f"rod on rails: emf {emf:.2f} V, I {I:.1f} A, retarding force {F:.2f} N, F v = {F * v:.2f} W, I^2 R = {I**2 * R:.2f} W")

N, A, B0, w = 100, 0.01, 0.5, 2 * np.pi * 50
t = np.linspace(0, 0.04, 801)
flux = N * B0 * A * np.cos(w * t)
emf_gen = ind.emf_from_flux(t, flux)
print(f"generator: peak emf from -dPhi/dt {np.abs(emf_gen).max():.1f} V, formula N B A omega = {N * B0 * A * w:.1f} V, rms {N * B0 * A * w / np.sqrt(2):.1f} V")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t * 1e3, flux / (N * B0 * A), label="flux linkage / (N B A)")
ax.plot(t * 1e3, emf_gen / (N * B0 * A * w), label="emf / (N B A omega)")
ax.set_xlabel("t [ms]"); ax.legend(loc="upper right"); ax.set_title("the generator emf peaks when the flux passes through zero")
plt.show()

**Worked example 4 (a loop moving through a region of field).** A rectangular loop, width $a = 0.1\,$m along $x$ and height $b = 0.05\,$m along $y$, moves along $+x$ at $v = 2\,$m/s. A field $B_z = B_0 = 0.5\,$T fills the strip $0 < x < 0.3\,$m and is zero outside, with edges smoothed over a few millimetres so that the field is a smooth function. Orient the loop counter-clockwise seen from $+z$, so that its normal is $+\hat{\mathbf{z}}$ and flux along $+z$ counts as positive.

By hand, $\mathbf{v}\times\mathbf{B} = v\hat{\mathbf{x}}\times B\hat{\mathbf{z}} = -vB\,\hat{\mathbf{y}}$. Only the two sides parallel to $y$ contribute to $\oint(\mathbf{v}\times\mathbf{B})\cdot d\mathbf{l}$. On the right side the path runs along $+y$ and gives $-vB(x_r)\,b$. On the left side it runs along $-y$ and gives $+vB(x_l)\,b$. So
$$\mathcal{E} = vb\,\big(B(x_l) - B(x_r)\big).$$
While the loop enters the field, its right side is in the field and its left side is not: $\mathcal{E} = -vbB_0 = -2\times0.05\times0.5 = -0.05\,$V. While it is wholly inside, both sides see the same field and $\mathcal{E} = 0$. While it leaves, $\mathcal{E} = +0.05\,$V.

The flux rule says the same. The flux is $\Phi = b\int_{x_l}^{x_r}B\,dx$, so $d\Phi/dt = vb\,(B(x_r) - B(x_l))$ and $-d\Phi/dt$ is the expression above. A negative emf drives a clockwise current seen from $+z$. That current makes a field along $-z$ inside the loop, opposing the growing flux along $+z$, as Lenz's law requires.

The code computes the flux as a time series, differentiates it with `ind.emf_from_flux`, and compares the result with the line integral of $\mathbf{v}\times\mathbf{B}$ around the loop at every instant.

In [ ]:
B0, w_edge, x1, x2 = 0.5, 0.005, 0.0, 0.3                 # field strip and edge width [m]
a, b, v = 0.1, 0.05, 2.0
Bz = lambda x: B0 * 0.5 * (np.tanh((x - x1) / w_edge) - np.tanh((x - x2) / w_edge))

t = np.linspace(0.0, 0.35, 1401)
x_left = -0.15 + v * t
s = np.linspace(0, a, 2001)
flux = np.array([b * np.trapezoid(Bz(xl + s), s) for xl in x_left])
emf_flux = ind.emf_from_flux(t, flux)

def emf_motional(xl, n=400):
    # counter-clockwise rectangle seen from +z, closed polyline
    corners = np.array([[xl, 0, 0], [xl + a, 0, 0], [xl + a, b, 0], [xl, b, 0], [xl, 0, 0]])
    path = np.concatenate([np.linspace(corners[k], corners[k + 1], n, endpoint=False) for k in range(4)] + [corners[-1:]])
    vxB = lambda p: np.cross(np.array([v, 0.0, 0.0]), np.stack([0 * p[:, 0], 0 * p[:, 0], Bz(p[:, 0])], axis=-1))
    return fields.line_integral(vxB, path)

emf_line = np.array([emf_motional(xl) for xl in x_left])
diff_loop = np.abs(emf_flux - emf_line)[1:-1].max() / (v * b * B0)
print(f"entering: emf = {emf_line[np.argmin(emf_line)]:+.4f} V   inside: {emf_line[np.argmin(np.abs(x_left - 0.1))]:+.1e} V   leaving: {emf_line.max():+.4f} V")
print(f"largest difference between -dPhi/dt and the v x B line integral: {diff_loop:.1e} of v b B0")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(t, flux * 1e3); ax1.set_xlabel("t [s]"); ax1.set_ylabel("flux [mWb]"); ax1.set_title("flux through the moving loop")
ax2.plot(t, emf_flux, lw=3, alpha=0.4, label="-dPhi/dt"); ax2.plot(t, emf_line, "k--", label="loop integral of v x B")
ax2.set_xlabel("t [s]"); ax2.set_ylabel("emf [V]"); ax2.legend(); ax2.set_title("two routes, one emf")
plt.tight_layout(); plt.show()

### 3.3 Faraday's law (section 14.3)

Now hold the circuit still and change the field. Faraday's experiments showed that the flux rule still holds:
$$\mathcal{E} = \oint_C\mathbf{E}\cdot d\mathbf{l} = -\frac{d}{dt}\int_S\mathbf{B}\cdot d\mathbf{S},$$
where $C$ is the stationary loop and $S$ any surface bounded by it. The carriers are not moving, so the only force that can push them is an electric field, and the left side is its circulation. This is Faraday's law. It applies to any closed path in space, whether or not a wire lies along it; the wire only lets you notice the field by the current it drives.

**Orientation and sign.** The loop direction and the surface normal are tied by the right-hand rule: curl the fingers of the right hand along the direction of $C$ and the thumb gives the positive normal of $S$. A positive emf drives current along that direction.

**Lenz's law.** The induced current produces its own field, and that field opposes the change in flux that drove it. Opposes the change, not the flux: if the flux is falling, the induced current tries to keep it up.

**Worked example 5 (three sign cases).** In each case, look along the loop's axis from the side the magnet or coil is on.

1. The north pole of a magnet approaches the loop. Field lines leave a north pole, so the flux through the loop pointing away from the observer grows. The induced current must make a field pointing back towards the observer inside the loop, which means it circulates counter-clockwise as seen by the observer. The loop then acts like a magnet with its north face towards the approaching north pole, and repels it.
2. The same pole recedes. The flux away from the observer shrinks. The induced current tries to keep it, making a field pointing away from the observer inside the loop: clockwise as seen by the observer. The loop now attracts the magnet and holds it back.
3. A nearby coaxial coil carries a steady current that is suddenly switched off. The flux it made through the loop collapses. The induced current flows in the same direction as the vanished current, briefly keeping the flux up. Switching on gives a pulse the other way.

In all three cases the magnetic force on the induced current opposes the relative motion or the change, so work must be done to make it happen, and that work appears as heat in the loop.

**The induced field around a solenoid.** A long solenoid of radius $R$ with $n$ turns per metre carries a current $I(t)$, so inside it $B = \mu_0nI$ along the axis and outside $B = 0$ (primer M4.0b). By symmetry the induced $\mathbf{E}$ circulates around the axis, $\mathbf{E} = E_\phi(r)\,\hat{\boldsymbol{\phi}}$. Apply Faraday's law to a circle of radius $r$ about the axis, traversed in the $+\phi$ direction:

- Inside, $r < R$: $2\pi rE_\phi = -\pi r^2\,\dot B$, so $E_\phi = -\dfrac{r}{2}\dot B$.
- Outside, $r > R$: the flux is all inside the solenoid, $\pi R^2B$, so $2\pi rE_\phi = -\pi R^2\dot B$ and $E_\phi = -\dfrac{R^2}{2r}\dot B$.

The induced field exists outside the solenoid, where $B$ is zero at all times. A charge there feels a force although no magnetic field ever reaches it. The field is what the flux rule demands for the circulation, not a local response to $\mathbf{B}$ at that point.

**Worked example 6 (numbers).** $n = 1000\,\mathrm{m^{-1}}$, $R = 0.02\,$m, current rising at $\dot I = 100\,$A/s. Then $\dot B = \mu_0n\dot I = 4\pi\times10^{-7}\times1000\times100 = 0.1257\,$T/s. At the winding, $|E_\phi| = (R/2)\dot B = 0.01\times0.1257 = 1.257\times10^{-3}\,$V/m. At $r = 2R$, $|E_\phi| = (R/4)\dot B = 6.28\times10^{-4}\,$V/m. A single loop of wire around the solenoid, of any shape and size, has emf $\pi R^2\dot B = 1.579\times10^{-4}\,$V. The minus sign makes $E_\phi$ point against the direction of the solenoid current, so a ring around the solenoid carries a current opposite to the winding current while that current rises.

In [ ]:
n_turns, Rs, dIdt = 1000.0, 0.02, 100.0
dBdt = MU0 * n_turns * dIdt
r = np.linspace(0.001, 0.08, 400)
E_phi = np.where(r < Rs, -r / 2 * dBdt, -Rs**2 / (2 * r) * dBdt)
print(f"dB/dt = {dBdt:.4f} T/s, |E| at r = R: {Rs / 2 * dBdt:.3e} V/m, at r = 2R: {Rs / 4 * dBdt:.3e} V/m")
print(f"emf of any loop around the solenoid: {np.pi * Rs**2 * dBdt:.3e} V")
for rr in (0.03, 0.05, 0.08):
    print(f"   circle of radius {rr} m: 2 pi r E_phi = {2 * np.pi * rr * (-Rs**2 / (2 * rr) * dBdt):.3e} V")
plt.plot(r * 100, -E_phi * 1e3); plt.axvline(Rs * 100, color="grey", ls=":")
plt.xlabel("r [cm]"); plt.ylabel("|E_phi| [mV/m]"); plt.title("induced field: linear inside, 1/r outside where B = 0"); plt.show()

**Worked example 7 (a magnet through a loop).** A small magnet, modelled as a point dipole $\mathbf{m} = m\hat{\mathbf{z}}$, moves along the axis of a loop of radius $R$ at speed $v$. Put the loop at $z = 0$ and the dipole at $z(t) = vt$. The flux of a dipole through a coaxial loop is easiest to get from reciprocity, which primer 00b proves: the flux of the dipole through the loop equals $m$ times the field that a unit current in the loop would make at the dipole. From primer M4.0a, a loop with unit current makes $B_z = \mu_0R^2/2(R^2 + z^2)^{3/2}$ on its axis, so
$$\Phi(z) = \frac{\mu_0mR^2}{2(R^2 + z^2)^{3/2}}.$$
It is largest, $\mu_0m/2R$, when the dipole sits in the plane of the loop. Differentiating with the chain rule,
$$\mathcal{E} = -\frac{d\Phi}{dt} = -\frac{d\Phi}{dz}\,v = \frac{3\mu_0mR^2\,zv}{2(R^2 + z^2)^{5/2}}.$$
The emf is negative while the magnet approaches from below ($z < 0$), passes through zero as it crosses the loop, and is positive as it leaves: the induced current reverses when the flux stops growing and starts to shrink. The extremes are at $z = \pm R/2$.

With $m = 1\,\mathrm{A\,m^2}$ (a strong magnet of about a cubic centimetre), $R = 2\,$cm and $v = 1\,$m/s: peak flux $\mu_0m/2R = 3.14\times10^{-5}\,$Wb, and peak emf at $z = R/2$
$$\frac{3\mu_0mv}{4R^2(5/4)^{5/2}} = \frac{3\times1.257\times10^{-6}}{4\times4\times10^{-4}\times1.747} = 1.35\times10^{-3}\,\mathrm{V}.$$

The code computes the flux three ways: from the closed form, with `ind.disk_flux` of the dipole field, and as the circulation of the dipole's vector potential $\mathbf{A} = (\mu_0/4\pi)\,\mathbf{m}\times\hat{\mathbf{r}}/r^2$ around the rim (primer M4.0b: $\Phi = \oint\mathbf{A}\cdot d\mathbf{l}$). The disk route cannot be used while the dipole is close to the plane of the loop, because a point dipole's field is singular at the dipole and the singular part carries flux through the disk exactly where the sampling is worst. The rim integral never comes close to the dipole and works at every position.

In [ ]:
m, R, v = 1.0, 0.02, 1.0
flux_closed = lambda z: MU0 * m * R**2 / (2 * (R**2 + z**2) ** 1.5)
emf_closed = lambda z: 3 * MU0 * m * R**2 * z * v / (2 * (R**2 + z**2) ** 2.5)

th = np.linspace(0, 2 * np.pi, 400, endpoint=False)
rim_mid, rim_dl = bs.segments(np.stack([R * np.cos(th), R * np.sin(th), 0 * th], axis=-1))
def flux_rim(zm):
    r = rim_mid - np.array([0.0, 0.0, zm])
    A = MU0 / (4 * np.pi) * np.cross([0.0, 0.0, m], r) / np.linalg.norm(r, axis=1, keepdims=True) ** 3
    return np.sum(A * rim_dl)

def flux_disk(zm):
    return ind.disk_flux(lambda p: bs.dipole_B([0.0, 0.0, m], p - np.array([0.0, 0.0, zm])), (0, 0, 0), (0, 0, 1), R)

t = np.linspace(-0.1, 0.1, 2001)
z = v * t
flux = np.array([flux_rim(zz) for zz in z])
emf = ind.emf_from_flux(t, flux)
print(f"peak flux {flux.max():.4e} Wb   closed form mu0 m / 2R = {MU0 * m / (2 * R):.4e} Wb")
for zz in (-0.06, -0.01, 0.0, 0.02):
    d = f"{flux_disk(zz):.5e}" if abs(zz) >= 0.3 * R else "  (dipole too close to the disk)"
    print(f"z = {zz:+.2f} m: closed form {flux_closed(zz):.5e}   rim integral of A {flux_rim(zz):.5e}   disk_flux {d}")
diff_dip = np.abs(emf - emf_closed(z))[1:-1].max() / np.abs(emf_closed(z)).max()
print(f"peak emf {emf.min():+.3e} V at z = {z[np.argmin(emf)]:+.4f} m and {emf.max():+.3e} V at z = {z[np.argmax(emf)]:+.4f} m")
print(f"largest difference from the hand-differentiated emf: {diff_dip:.1e} of the peak")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(z * 100, flux * 1e6); ax1.set_xlabel("magnet position z [cm]"); ax1.set_ylabel("flux [uWb]"); ax1.set_title("flux through the loop")
ax2.plot(z * 100, emf * 1e3, lw=3, alpha=0.4, label="-dPhi/dt, numerical"); ax2.plot(z * 100, emf_closed(z) * 1e3, "k--", label="by hand")
ax2.axhline(0, color="grey", lw=0.5); ax2.set_xlabel("magnet position z [cm]"); ax2.set_ylabel("emf [mV]"); ax2.legend(); ax2.set_title("the emf reverses as the magnet passes")
plt.tight_layout(); plt.show()

**Animation.** A small magnet moves along the axis of a conducting loop, through it and out the other side, with its field lines threading the loop. The readout tracks the flux through the loop, and the plot beside it draws the flux and the emf against time as the magnet moves. Watch the induced-current marker on the loop: it points one way while the flux grows, passes through zero when the magnet is in the plane of the loop, and reverses while the flux shrinks. In both halves the current opposes the change, which is Lenz's law.

In [ ]:
from emlib import show
show.animation("m6_waves", "FaradayInduction")

### 3.4 The fine print of the flux rule

**Any surface will do.** The flux in Faraday's law is through "a surface bounded by the loop", and there are infinitely many such surfaces: a flat disk, a bowl, a long sock. They all give the same flux because $\nabla\cdot\mathbf{B} = 0$ (primer M4.0b). Two surfaces with the same rim together enclose a volume, the net flux out of any closed surface is zero, so whatever enters through one leaves through the other. Without this the flux rule would be ambiguous. The code checks it for the dipole field and three surfaces with the same rim: the flat disk, a hemisphere bulging above it, and an open cylinder with a lid.

**Orientation, once more.** Reversing the direction around the loop reverses the normal, which flips the sign of both the flux and the emf. The physics does not change; the sign of a current is only meaningful relative to the chosen direction. The rule to remember is that a positive emf pushes current in the direction that, by the right-hand rule, makes a field along the positive normal.

**When the naive flux rule fails.** The flux rule assumes that "the circuit" is a well-defined curve that moves with the material carrying the current. Some devices break that assumption. In the Faraday disk, or homopolar generator, a metal disk spins in a uniform field parallel to its axis, and sliding contacts touch it at the centre and at the rim. The flux through the circuit formed by the contacts, the disk and the external wire never changes, yet an emf appears. The force law explains it at once: a carrier at radius $r$ moves at $\omega r$ across the field, so the force per charge along the radius is $\omega rB$ and
$$\mathcal{E} = \int_0^R\omega rB\,dr = \tfrac12B\omega R^2.$$
The reverse also happens: rocking a switch between two contact points can change the flux through "the circuit" abruptly without producing any emf. The safe statement is always the force law, $\mathcal{E} = \oint(\mathbf{E} + \mathbf{v}\times\mathbf{B})\cdot d\mathbf{l}$, with $\mathbf{v}$ the velocity of the material. The flux rule is a convenient consequence of it when the circuit is a closed wire moving as a whole.

**Worked example 8 (a Faraday disk).** A copper disk of radius $R = 0.1\,$m spins at 3000 revolutions per minute, $\omega = 314.2\,\mathrm{s^{-1}}$, in a field $B = 0.5\,$T along its axis. The emf between centre and rim is $\tfrac12\times0.5\times314.2\times0.01 = 0.785\,$V. Homopolar generators give low voltages but enormous currents, because the disk's resistance is tiny.

In [ ]:
# The same flux through three surfaces with one rim: the circle of radius R at z = 0.
m, R, zd = 1.0, 0.02, -0.03                                   # dipole 3 cm below the loop
Bd = lambda p: bs.dipole_B([0.0, 0.0, m], p - np.array([0.0, 0.0, zd]))

flux_flat = ind.disk_flux(Bd, (0, 0, 0), (0, 0, 1), R, nr=200, nphi=200)

nt, nph = 300, 300                                            # hemisphere above the rim, normal r_hat (upwards)
th = (np.arange(nt) + 0.5) * (np.pi / 2) / nt; ph = (np.arange(nph) + 0.5) * 2 * np.pi / nph
TH, PH = np.meshgrid(th, ph, indexing="ij")
nrm = np.stack([np.sin(TH) * np.cos(PH), np.sin(TH) * np.sin(PH), np.cos(TH)], axis=-1).reshape(-1, 3)
dA = (R**2 * np.sin(TH) * (np.pi / 2 / nt) * (2 * np.pi / nph)).ravel()
flux_hemi = np.sum(np.einsum("ij,ij->i", Bd(R * nrm), nrm) * dA)

H, nz = 0.05, 400                                             # open cylinder of height H with a lid
zc = (np.arange(nz) + 0.5) * H / nz
ZC, PC = np.meshgrid(zc, ph, indexing="ij")
side_n = np.stack([np.cos(PC), np.sin(PC), 0 * PC], axis=-1).reshape(-1, 3)
side_p = np.stack([R * np.cos(PC), R * np.sin(PC), ZC], axis=-1).reshape(-1, 3)
flux_side = np.sum(np.einsum("ij,ij->i", Bd(side_p), side_n)) * R * (2 * np.pi / nph) * (H / nz)
flux_cyl = flux_side + ind.disk_flux(Bd, (0, 0, H), (0, 0, 1), R, nr=200, nphi=200)

print(f"flat disk {flux_flat:.6e} Wb   hemisphere {flux_hemi:.6e} Wb   cylinder with lid {flux_cyl:.6e} Wb")
print(f"closed form for the disk: {MU0 * m * R**2 / (2 * (R**2 + zd**2) ** 1.5):.6e} Wb")
print(f"Faraday disk: emf = B omega R^2 / 2 = {0.5 * 0.5 * 2 * np.pi * 50 * 0.1**2:.3f} V")

### 3.5 Faraday's law in differential form (section 14.4)

Apply Stokes' theorem (primer M1.0d) to the left side of Faraday's law for a fixed loop: $\oint_C\mathbf{E}\cdot d\mathbf{l} = \int_S(\nabla\times\mathbf{E})\cdot d\mathbf{S}$. The right side is $-\int_S\partial\mathbf{B}/\partial t\cdot d\mathbf{S}$ for a stationary surface. The two surface integrals agree for every surface, however small, so the integrands are equal:
$$\nabla\times\mathbf{E} = -\frac{\partial\mathbf{B}}{\partial t}.$$
This is Faraday's law at a point. It says that a changing magnetic field is a source of circulation for the electric field, just as a current is a source of circulation for the magnetic field (Ampere's law). When nothing changes, it reduces to $\nabla\times\mathbf{E} = 0$, the statement that made the electrostatic potential possible. Static electricity is therefore the special case of slowly changing fields, not a separate theory.

**Worked example 9 (the curl of the solenoid's induced field).** For a $\phi$-directed field the $z$ component of the curl in cylindrical coordinates is $\frac1r\frac{d}{dr}(rE_\phi)$ (primer M1.0d). Inside the solenoid, $rE_\phi = -r^2\dot B/2$, so the curl is $\frac1r(-r\dot B) = -\dot B$, as Faraday's law requires. Outside, $rE_\phi = -R^2\dot B/2$ is constant, so the curl is zero, as it must be where $\mathbf{B}$ does not change. Yet the circulation around any loop that encloses the solenoid is $-\pi R^2\dot B$, not zero. This is the wire-field puzzle of primer M1.0c in a new setting: a field with zero curl everywhere outside a region can still circulate around that region, because Stokes' theorem applied to such a loop needs a surface that crosses the region, where the curl is not zero.

**A field like any other.** The induced field pushes on charges whether or not a wire is present. The betatron uses this: electrons travel in an evacuated ring around a magnet whose field is ramped up, and the induced electric field along the ring accelerates them on every lap. The same rising field also bends them, and the machine works when the field at the orbit is half the average field inside it, so that the orbit radius stays fixed while they speed up.

**The potentials.** Since $\mathbf{B} = \nabla\times\mathbf{A}$ (primer M4.0b), Faraday's law reads $\nabla\times(\mathbf{E} + \partial\mathbf{A}/\partial t) = 0$. A field with zero curl is a gradient, so
$$\mathbf{E} = -\nabla V - \frac{\partial\mathbf{A}}{\partial t}.$$
The electric field has two parts: the familiar gradient of a potential, made by charges, and minus the rate of change of the vector potential, made by changing currents. The second part is the induced field. With no charges about, $\mathbf{E} = -\partial\mathbf{A}/\partial t$, which gives a direct way to compute it: find $\mathbf{A}$ with the Biot-Savart sum for unit current, and multiply by $-dI/dt$.

**Worked example 10 (the solenoid from $\mathbf{A}$).** Inside a long solenoid $A_\phi = \mu_0nIr/2$ (the flux $\mu_0nI\,\pi r^2$ divided by the circumference $2\pi r$, primer M4.0b). Then $E_\phi = -\partial A_\phi/\partial t = -\mu_0n\dot Ir/2 = -(r/2)\dot B$, the result of worked example 6 without any loop argument. The code builds a helix of 1000 turns, 1 m long and 2 cm in radius, computes $\mathbf{A}$ at points inside and outside with `bs.wire_A`, and multiplies by $-\dot I$.

In [ ]:
N, Lsol, Rs, dIdt = 1000, 1.0, 0.02, 100.0
mid, dl = bs.segments(bs.helix(Rs, Lsol, N, 40), closed=False)
r_in = np.array([0.1, 0.3, 0.5, 0.7]) * Rs
r_out = np.array([1.3, 1.6, 2.0, 2.5, 3.0]) * Rs
r_all = np.concatenate([r_in, r_out])
A_unit = bs.wire_A(mid, dl, 1.0, np.stack([r_all, 0 * r_all, 0 * r_all], axis=-1))   # points on +x, so A_phi = A_y
E_from_A = -dIdt * A_unit[:, 1]
dBdt = MU0 * (N / Lsol) * dIdt
E_closed = np.where(r_all < Rs, -r_all / 2 * dBdt, -Rs**2 / (2 * r_all) * dBdt)
for rr, e1, e2 in zip(r_all, E_from_A, E_closed):
    print(f"r = {rr / Rs:.1f} R:  -dA/dt = {e1:+.4e} V/m   closed form {e2:+.4e} V/m   ratio {e1 / e2:.4f}")
rr_plot = np.linspace(0.01, 3, 300) * Rs
plt.plot(rr_plot / Rs, np.where(rr_plot < Rs, rr_plot / 2, Rs**2 / (2 * rr_plot)) * dBdt * 1e3, "k-", label="long-solenoid closed form")
plt.plot(r_all / Rs, -E_from_A * 1e3, "o", label="-dA/dt from the helix")
plt.xlabel("r / R"); plt.ylabel("|E_phi| [mV/m]"); plt.legend(); plt.title("the induced field from the vector potential"); plt.show()

The inside points agree to about a tenth of a percent. The small excess is the error of the one-point-per-segment sum with 40 segments per turn: with 80 and 160 segments per turn the ratio inside falls to 0.9997 and 0.9993, approaching 0.9992, the factor $L/\sqrt{L^2 + 4R^2}$ by which a solenoid of finite length falls short of the infinite one at its centre. Outside, the helix gives a field that falls slowly below the closed form, by about one percent at $3R$. That is physics, not error: the closed form is for an infinite solenoid, whose flux is entirely inside. A real solenoid of finite length returns part of its flux through the space outside, so a large circle encloses slightly less net flux than $\pi R^2B$, and the induced field around it is correspondingly smaller.

### 3.6 Eddy currents and applications

**Eddy currents.** Put a solid conductor in a changing magnetic field and the induced electric field drives currents in closed loops inside it. These eddy currents dissipate power at the rate $\sigma E^2$ per unit volume (primer M5.0a), and by Lenz's law their own field opposes the change. Both effects matter in practice.

- **Magnetic braking.** A strong magnet dropped down a vertical copper tube falls slowly. As it moves, the flux through every ring of the tube ahead of it grows and behind it shrinks, and the eddy currents in those rings push back on the magnet. The drag force grows in proportion to the speed and to the conductivity, so the magnet quickly reaches a terminal speed at which the drag equals its weight. Doubling the wall thickness doubles the drag and halves the terminal speed. Trains and roller coasters use the same principle as frictionless brakes.
- **Transformer cores.** A transformer needs iron to guide the flux, but a solid iron core would be a short-circuited turn around that flux, and eddy currents would waste power and heat it. Cores are built from thin insulated sheets, laminations, oriented so that the eddy currents can only flow in thin layers. Worked example 12 shows why that helps so much.
- **Induction cooking.** A coil under the cooktop carries current at tens of kilohertz. The changing field drives eddy currents in the bottom of an iron or steel pan, which heats up, while the glass top stays cool because it does not conduct.
- **Metal detectors.** A transmitter coil sends a pulse or an oscillating field into the ground. Eddy currents induced in a buried metal object make their own field, which a receiver coil picks up after the transmitted pulse has ended.

**Worked example 11 (eddy loss in a thin disk).** A thin disk of radius $R$, thickness $d$ and conductivity $\sigma$ sits in a uniform field $B(t) = B_0\sin\omega t$ along its axis. Treat the induced field as that of the applied field alone, which is valid when the eddy currents are too weak to change $B$ much (the disk thin compared with the skin depth, primer 00c). By the solenoid argument with $R$ replaced by $r$, the induced field at radius $r$ has magnitude $E = (r/2)\,|\dot B|$. The power density is $\sigma E^2 = \sigma r^2\dot B^2/4$, and the instantaneous power is
$$P = \int_0^R\frac{\sigma r^2\dot B^2}{4}\,2\pi rd\,dr = \frac{\pi\sigma d\,\dot B^2R^4}{8}.$$
With $\dot B = B_0\omega\cos\omega t$ and the average of $\cos^2$ equal to $\tfrac12$, the time-averaged power is
$$\langle P\rangle = \frac{\pi\sigma d\,\omega^2B_0^2R^4}{16}.$$
For an aluminium disk, $\sigma = 3.5\times10^7\,$S/m, $d = 1\,$mm, $R = 5\,$cm, in a 50 Hz field of amplitude $B_0 = 10\,$mT: $\langle P\rangle = 0.424\,$W. The $R^4$ is the important feature. Cut the disk into four quarter-size disks of radius $R/2$ and each dissipates $1/16$ as much, a quarter of the original in total. Smaller loops enclose less flux and have shorter paths.

**Worked example 12 (laminations).** A slab of thickness $t$ (along $x$), extending far in $y$ and $z$, sits in a uniform field $B_z(t)$. By symmetry the induced field points along $y$ and depends on $x$ only, and $\nabla\times\mathbf{E} = -\dot{\mathbf{B}}$ gives $dE_y/dx = -\dot B$, so $E_y = -x\dot B$ with $x$ measured from the mid-plane. The power per unit volume, averaged over the thickness, is $\sigma\dot B^2\langle x^2\rangle = \sigma\dot B^2t^2/12$, and averaged over a cycle
$$p = \frac{\sigma\omega^2B_0^2t^2}{24}.$$
The loss per unit volume grows as the square of the thickness. Split a 10 mm core into twenty 0.5 mm laminations and the loss falls by $20^2 = 400$. For electrical steel, $\sigma \approx 2\times10^6\,$S/m (silicon is added to lower it), $B_0 = 1\,$T, 50 Hz and $t = 0.5\,$mm: $p = 2\times10^6\times9.87\times10^4\times1\times2.5\times10^{-7}/24 = 2.06\times10^3\,\mathrm{W/m^3}$, about 0.27 W per kilogram. The formula assumes the sheet is thin compared with the skin depth, which holds for 0.5 mm laminations at 50 Hz but not for a solid 10 mm block, where the real loss is lower than $t^2$ predicts but still far too high.

In [ ]:
sigma, d, R, B0, w = 3.5e7, 1e-3, 0.05, 0.01, 2 * np.pi * 50
r = (np.arange(4000) + 0.5) * R / 4000
P_inst_per_Bdot2 = np.sum(sigma * r**2 / 4 * 2 * np.pi * r * d) * (R / 4000)       # integral of sigma E^2 dV per (dB/dt)^2
P_avg_num = P_inst_per_Bdot2 * (B0 * w) ** 2 / 2
P_avg_formula = np.pi * sigma * d * w**2 * B0**2 * R**4 / 16
print(f"eddy loss in the disk: numerical {P_avg_num:.4f} W, formula {P_avg_formula:.4f} W")
quarter = np.pi * sigma * d * w**2 * B0**2 * (R / 2) ** 4 / 16
print(f"four disks of radius R/2: {4 * quarter:.4f} W in total, {4 * quarter / P_avg_formula:.2f} of the single disk")

sig_steel, B0s, t_lam = 2e6, 1.0, 0.5e-3
x = (np.arange(2000) + 0.5) / 2000 * t_lam - t_lam / 2
p_num = sig_steel * np.mean(x**2) * (B0s * w) ** 2 / 2
p_formula = sig_steel * w**2 * B0s**2 * t_lam**2 / 24
print(f"lamination loss: numerical {p_num:.4e} W/m^3, formula {p_formula:.4e} W/m^3, {p_formula / 7650:.3f} W/kg")
print(f"a solid 10 mm block by the same formula: {(10e-3 / t_lam) ** 2:.0f} times more per unit volume")

### Exercise: the rod on rails

Replace the body of `rod_on_rails` with your own version. It takes the field $B$, the rail spacing $L$, the rod's velocity $v$ (positive to the right, negative to the left) and the resistance $R$, and returns the emf, the current and the magnetic force on the rod, with the force signed along the direction of positive $v$. The checks require the power balance $-Fv = I^2R$, and that the force always opposes the motion.

In [ ]:
def rod_on_rails(B, L, v, R):
    '''Return (emf, I, F) for a rod of length L sliding at velocity v on rails in a field B, closed by R.'''
    emf = B * L * v                        # replace from here with your own version
    I = emf / R
    F = -B * I * L                         # the force on the current opposes the motion
    return emf, I, F


for B, L, v, R in ((0.5, 0.2, 2.0, 0.1), (1.2, 0.05, -3.0, 2.0), (0.01, 1.0, 10.0, 0.5)):
    emf, I, F = rod_on_rails(B, L, v, R)
    print(f"B = {B}, L = {L}, v = {v:+}, R = {R}:  emf {emf:+.4f} V, I {I:+.4f} A, F {F:+.4e} N, -F v = {-F * v:.4e} W, I^2 R = {I**2 * R:.4e} W")
    assert np.isclose(-F * v, I**2 * R)
    assert np.sign(F) == -np.sign(v)
print("exercise checks passed")

## 4. Verify

Two independent checks with asserts. The emf of the loop moving through a field strip, from the time derivative of its flux, must equal the line integral of $\mathbf{v}\times\mathbf{B}$ around it at every instant (section 3.2). The induced field of the helix computed as $-\partial\mathbf{A}/\partial t$ must match the long-solenoid closed form inside to better than half a percent and outside to better than one and a half percent, the outside tolerance allowing for the finite length discussed in section 3.5. A third check confirms that the numerically differentiated emf of the moving magnet matches the hand derivative.

In [ ]:
print(f"moving loop: largest |(-dPhi/dt) - (v x B line integral)| = {diff_loop:.1e} of v b B0")
assert diff_loop < 1e-3

rel = np.abs(E_from_A / E_closed - 1)
print(f"solenoid: largest relative error inside {rel[:len(r_in)].max():.1e}, outside {rel[len(r_in):].max():.1e}")
assert rel[:len(r_in)].max() < 5e-3 and rel[len(r_in):].max() < 1.5e-2

print(f"magnet through the loop: largest emf difference {diff_dip:.1e} of the peak")
assert diff_dip < 1e-3
print("all verify checks passed")

## 5. Explore and rebuild

Explore. Drop the magnet of worked example 7 through a stack of 20 coaxial loops spaced 1 cm apart and connected in series, so that the emfs add. Plot the total emf as the magnet falls from rest under gravity, ignoring the braking. Then add the braking: if the loops are closed through a resistance $R$, each carries a current $\mathcal{E}_k/R$, and the magnet feels a force from each that you can get from the power balance ($Fv = \sum\mathcal{E}_k^2/R$). Integrate the motion with `emlib.integrators.rk4_ode` and find the terminal speed. How does it scale with $R$ and with $m$?

Rebuild. Close this notebook. On paper, derive the motional emf of the rod on rails both from the force on the carriers and from the flux rule, and show the power balance. Derive the induced field inside and outside a long solenoid with a changing current. Derive $\nabla\times\mathbf{E} = -\partial\mathbf{B}/\partial t$ from the integral form. In a blank editor, compute the flux of a moving dipole through a loop as a time series and differentiate it, then reproduce the plot of worked example 7.

## 6. Key results

| result | formula | holds when |
|---|---|---|
| emf | $\mathcal{E} = \oint\mathbf{f}\cdot d\mathbf{l}$, work per charge around a closed path | any circuit; electrostatic fields contribute zero |
| force law form | $\mathcal{E} = \oint(\mathbf{E} + \mathbf{v}\times\mathbf{B})\cdot d\mathbf{l}$ | $\mathbf{v}$ the velocity of the conducting material |
| motional emf | $\mathcal{E} = \oint(\mathbf{v}\times\mathbf{B})\cdot d\mathbf{l}$ | static field, moving circuit |
| rod on rails | $\mathcal{E} = BLv$, $I = BLv/R$, $F = B^2L^2v/R$ against $v$, $Fv = I^2R$ | uniform $B$ perpendicular to rails and rod |
| generator | $\mathcal{E} = NBA\omega\sin\omega t$ | flat coil rotating in uniform $B$ |
| magnetic flux | $\Phi_B = \int_S\mathbf{B}\cdot d\mathbf{S}$, the same for every surface with the same rim | $\nabla\cdot\mathbf{B} = 0$ |
| Faraday's law | $\oint_C\mathbf{E}\cdot d\mathbf{l} = -\dfrac{d}{dt}\int_S\mathbf{B}\cdot d\mathbf{S}$ | fixed loop; for a wire moving as a whole, $\mathcal{E} = -d\Phi_B/dt$ |
| Lenz's law | the induced current opposes the change in flux | always (energy conservation) |
| differential form | $\nabla\times\mathbf{E} = -\partial\mathbf{B}/\partial t$ | everywhere |
| fields from potentials | $\mathbf{E} = -\nabla V - \partial\mathbf{A}/\partial t$, $\mathbf{B} = \nabla\times\mathbf{A}$ | everywhere |
| induced field of a long solenoid | $E_\phi = -\tfrac{r}{2}\dot B$ inside, $-\tfrac{R^2}{2r}\dot B$ outside | infinite solenoid, $\dot B = \mu_0n\dot I$ |
| dipole on the axis of a loop | $\Phi = \dfrac{\mu_0mR^2}{2(R^2 + z^2)^{3/2}}$, $\mathcal{E} = \dfrac{3\mu_0mR^2zv}{2(R^2 + z^2)^{5/2}}$ | point dipole along the axis |
| Faraday disk | $\mathcal{E} = \tfrac12B\omega R^2$ | uniform $B$ along the axis, contacts at centre and rim |
| eddy loss, thin disk | $\langle P\rangle = \pi\sigma d\,\omega^2B_0^2R^4/16$ | disk thin compared with the skin depth |
| eddy loss, lamination | $p = \sigma\omega^2B_0^2t^2/24$ per unit volume | sheet thin compared with the skin depth |

## 7. Warm-up answers

1. An emf is the work per unit charge done by all forces on the carriers around a closed circuit. Only non-electrostatic forces contribute. A potential difference is defined between two points by the conservative electrostatic field and does not depend on the path; an emf belongs to a closed path.
2. $\mathcal{E} = BLv$, from the magnetic force $q\mathbf{v}\times\mathbf{B}$ on the carriers in the rod. The current in the rod feels a force $BIL$ against the motion, so whatever keeps the rod moving does work at the rate $BILv = I^2R$; that work becomes the heat in the resistor.
3. $\oint_C\mathbf{E}\cdot d\mathbf{l} = -\frac{d}{dt}\int_S\mathbf{B}\cdot d\mathbf{S}$: the flux of the magnetic field through any surface $S$ bounded by the loop $C$, with the normal tied to the direction of $C$ by the right-hand rule.
4. Counter-clockwise as seen from the magnet. The flux pointing away from the observer grows, so the induced current makes a field pointing back towards the magnet inside the loop; the loop's near face becomes a north pole and repels the magnet.
5. Its circulation around a loop enclosing changing flux is $-d\Phi_B/dt \neq 0$, while the circulation of any gradient is zero.
6. $\nabla\times\mathbf{E} = -\partial\mathbf{B}/\partial t$. When the magnetic field does not change, the right side is zero and $\nabla\times\mathbf{E} = 0$, which lets $\mathbf{E} = -\nabla V$.
7. A solid core is a conducting loop around the changing flux, so large eddy currents flow and heat it. Thin insulated sheets confine the eddy currents to thin layers, and the loss per unit volume falls as the square of the sheet thickness.